# Minimal PandasAI + Ollama Test

This notebook verifies that PandasAI can call a local Ollama model and compute statistics from a small dataframe.

Prerequisites:

```bash
ollama pull gemma4:e4b
uv venv --python 3.11 .venv-pandasai
source .venv-pandasai/bin/activate
pip install pandasai pandasai-litellm pandas ipykernel
python -m ipykernel install --user --name smolnalysis-pandasai --display-name "smolnalysis pandasai"
```

Use the `smolnalysis pandasai` kernel for this notebook. PandasAI currently targets Python `<=3.11`, while the main repo environment is Python 3.12.

In [1]:
import json
import os
import urllib.request

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e4b")

with urllib.request.urlopen(f"{OLLAMA_BASE_URL}/api/tags", timeout=5) as response:
    tags = json.loads(response.read().decode("utf-8"))

models = [model["name"] for model in tags.get("models", [])]
print("Ollama models:", models)

if OLLAMA_MODEL not in models:
    raise RuntimeError(f"{OLLAMA_MODEL!r} is not installed. Run: ollama pull {OLLAMA_MODEL}")

Ollama models: ['gemma4:e4b']


In [2]:
import pandasai as pai
from pandasai_litellm.litellm import LiteLLM

llm = LiteLLM(
    model=f"ollama_chat/{OLLAMA_MODEL}",
    api_base=OLLAMA_BASE_URL,
    api_key="ollama",
)

pai.config.set({
    "llm": llm,
    "temperature": 0,
})

In [3]:
df = pai.DataFrame({
    "city": ["Munich", "Berlin", "Hamburg", "Munich", "Berlin", "Hamburg"],
    "year": [2023, 2023, 2023, 2024, 2024, 2024],
    "population_millions": [1.51, 3.76, 1.89, 1.52, 3.78, 1.90],
    "bike_count": [5200, 8700, 4100, 6100, 9300, 4550],
})

df

,city,year,population_millions,bike_count
0,Munich,2023,1.51,5200
1,Berlin,2023,3.76,8700
2,Hamburg,2023,1.89,4100
3,Munich,2024,1.52,6100
4,Berlin,2024,3.78,9300
5,Hamburg,2024,1.90,4550


In [4]:
response = df.chat(
    "Return the mean bike_count by city as a compact table, then name the city with the highest mean."
)

response

DataFrameResponse(type='dataframe', value=      city  mean_bike_count
0   Berlin           9000.0
1   Munich           5650.0
2  Hamburg           4325.0)

In [5]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "training" / "scripts" / "open_data_tools.py").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / "training" / "scripts"))

from open_data_tools import DataframeStore, retrieve_open_data, search_open_data

store = DataframeStore(repo_root / ".tmp" / "pandasai_ckan_data")

## Real CKAN Dataset

This uses the repo's CKAN helper functions to search Munich Open Data, retrieve a parseable resource, and pass the resulting dataframe to PandasAI.

In [6]:
search_result = search_open_data("Vornamen", limit=5)

candidates = [candidate for candidate in search_result["candidates"] if candidate["resources"]]
if not candidates:
    raise RuntimeError("No parseable CKAN candidates found for the search query.")

selected_package = candidates[0]
selected_resource = selected_package["resources"][0]

selected_package["package_title"], selected_resource

('Vornamen von Nulljährigen München',
 {'resource_id': '6388c83a-266d-437c-824a-7bbcb7ceec63',
  'resource_name': 'Vornamen 2025',
  'resource_format': 'csv',
  'datastore_active': True})

In [7]:
profile = retrieve_open_data(
    package_id_or_name=selected_package["package_name"],
    resource_id=selected_resource["resource_id"],
    limit=500,
    store=store,
)

ckan_df = store.load(profile["dataframe_id"])
for column in ckan_df.columns:
    numeric = pd.to_numeric(ckan_df[column], errors="coerce")
    if numeric.notna().mean() > 0.8:
        ckan_df[column] = numeric

print(profile["metadata"])
print(f"Rows: {len(ckan_df)}, columns: {len(ckan_df.columns)}")
ckan_df.head()

{'package_id': '99ad40ec-9d7b-4a2e-87eb-9bac783fb57a', 'package_name': 'vornamen-von-neugeborenen', 'package_title': 'Vornamen von Nulljährigen München', 'resource_id': '6388c83a-266d-437c-824a-7bbcb7ceec63', 'resource_name': 'Vornamen 2025', 'resource_format': 'csv', 'filter_mode': 'ckan_datastore', 'server_filter_supported': True, 'source': 'datastore_search'}
Rows: 500, columns: 4


,_id,vorname,anzahl,geschlecht
0,1,Felix,89,m
1,2,Anton,87,m
2,3,Emma,81,w
3,4,Emil,80,m
4,5,Clara,79,w


In [8]:
ckan_pai_df = pai.DataFrame(ckan_df)

ckan_response = ckan_pai_df.chat(
    "Inspect this real CKAN dataset. Return row count, column names, missing-value counts, "
    "and one useful grouped statistic based on the available columns."
)

ckan_response

--- 1. Dataset Overview (Row Count & Column Names) ---
Row Count: 500

--- 2. Missing Value Counts ---
Missing Value Counts:
- _id: 0 missing values
- vorname: 0 missing values
- anzahl: 0 missing values
- geschlecht: 0 missing values

--- 3. Grouped Statistic (Average 'anzahl' by Gender) ---


<string>:23: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead


--- 1. Dataset Overview (Row Count & Column Names) ---
--- 1. Dataset Overview (Row Count & Column Names) ---
Row Count: 500

--- 2. Missing Value Counts ---
Missing Value Counts:
- _id: 0
- vorname: 0
- anzahl: 0
- geschlecht: 0

--- 3. Grouped Statistic (Average 'anzahl' by Gender) ---


<string>:23: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead


KeyboardInterrupt: 